# Relatório Lab 4 — Parte A: Filtragem Espacial Passa-Baixa

**Autores:** Allan Hirose Pires de Souza  
**Data de realização dos experimentos:** Março de 2026  
**Data de publicação do relatório:** 10 de Março de 2026  

---

## 1. Introdução

Neste laboratório (Parte A), exploramos os **filtros passa-baixa** aplicados ao processamento digital de imagens. Esses filtros atenuam as altas frequências espaciais (bordas, ruídos, detalhes finos) e preservam as baixas frequências (regiões homogêneas, variações suaves de intensidade).

Os filtros estudados foram:
- **Filtro de Média** (Box Filter)
- **Filtro Gaussiano**
- **Filtro de Mediana**
- **Filtro Bilateral**

Cada filtro foi avaliado em imagens limpas e em imagens degradadas por dois tipos de ruído:
- **Ruído Gaussiano** (aditivo, contínuo)
- **Ruído Sal-e-Pimenta** (impulsivo, pontual)

A avaliação quantitativa foi feita com as métricas **PSNR** (Peak Signal-to-Noise Ratio) e **SSIM** (Structural Similarity Index).

---

## 2. Materiais e Métodos

### 2.1 Imagens utilizadas
- **Foto do grupo completo** (`../Lab3/imagem.jpeg`)
- **Avatar do grupo** (`../Lab3/avatar.jpeg`)
- **Foto individual de um membro** (mesma imagem, convertida para escala de cinza quando necessário)

### 2.2 Kernels avaliados
Os tamanhos de kernel testados foram: **3×3**, **5×5**, **11×11** e **29×29**.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
# Funcao auxiliar para visualizar imagens lado a lado
# -------------------------------------------------------
def plot_images(images, titles, rows=1, cols=None, figsize=(16, 5), cmap=None):
    if cols is None:
        cols = len(images)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows * cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# -------------------------------------------------------
# Carregamento das imagens
# -------------------------------------------------------
img_grupo = cv2.imread('../Lab3/imagem.jpeg')
img_avatar = cv2.imread('../Lab3/avatar.jpeg')

if img_grupo is None:
    print("[AVISO] imagem.jpeg nao encontrada — usando placeholder")
    img_grupo = np.random.randint(50, 200, (400, 600, 3), dtype=np.uint8)
if img_avatar is None:
    print("[AVISO] avatar.jpeg nao encontrado — usando placeholder")
    img_avatar = np.random.randint(50, 200, (400, 400, 3), dtype=np.uint8)

img_grupo_rgb  = cv2.cvtColor(img_grupo,  cv2.COLOR_BGR2RGB)
img_avatar_rgb = cv2.cvtColor(img_avatar, cv2.COLOR_BGR2RGB)

print(f"img_grupo  shape: {img_grupo.shape}")
print(f"img_avatar shape: {img_avatar.shape}")
plot_images([img_grupo_rgb, img_avatar_rgb], ['Foto do Grupo', 'Avatar do Grupo'], figsize=(12, 5))


## 3. Parte A — Item (a): Filtros em Imagens Limpas

Aplicamos os quatro filtros passa-baixa nas imagens sem ruído, com kernels de tamanhos variados.  
O objetivo é observar o efeito visual de cada filtro à medida que o kernel cresce.


In [ ]:
kernels = [3, 5, 11, 29]

# Gera uma grade: cada linha é um kernel, cada coluna é um filtro
fig, axes = plt.subplots(len(kernels), 4, figsize=(16, 4 * len(kernels)))
filtros_nomes = ['Media', 'Gaussiano', 'Mediana', 'Bilateral']

for row, k in enumerate(kernels):
    sigma = 0  # cv2 calcula automaticamente

    blur_media    = cv2.blur(img_grupo, (k, k))
    blur_gauss    = cv2.GaussianBlur(img_grupo, (k if k % 2 == 1 else k+1, k if k % 2 == 1 else k+1), sigma)
    blur_mediana  = cv2.medianBlur(img_grupo, k if k % 2 == 1 else k+1)
    blur_bilateral= cv2.bilateralFilter(img_grupo, k, 75, 75)

    filtrados = [blur_media, blur_gauss, blur_mediana, blur_bilateral]

    for col, (img_f, nome) in enumerate(zip(filtrados, filtros_nomes)):
        axes[row, col].imshow(cv2.cvtColor(img_f, cv2.COLOR_BGR2RGB))
        axes[row, col].set_title(f'{nome} k={k}', fontsize=9)
        axes[row, col].axis('off')

plt.suptitle('Filtros Passa-Baixa — Imagem Limpa (Foto do Grupo)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Discussão — Filtros em Imagem Limpa:**

- **Filtro de Média:** O mais agressivo em termos de borramento. Com kernel 29×29, praticamente elimina todos os detalhes finos. Trata todos os vizinhos com igual peso, sem qualquer distinção de borda.
- **Filtro Gaussiano:** Produz um borramento mais natural e progressivo. A contribuição dos pixels decresce conforme a distância ao centro (distribuição em sino), o que gera transições mais suaves e menos artefatos de bloco.
- **Filtro de Mediana:** Por ser um filtro estatístico de ordem, é não-linear. Em imagens limpas, pode parecer similar ao de média, mas é muito mais seletivo — não cria novos valores na imagem, apenas reorganiza os existentes.
- **Filtro Bilateral:** O diferencial é a preservação de bordas: ele pondera os pixels vizinhos tanto pela distância espacial quanto pela similaridade de intensidade. Em imagens limpas, o efeito visual é de um borramento suave que respeita as arestas principais da imagem.


## 3. Parte A — Item (b): Adição de Ruído e Filtragem

Nesta etapa introduzimos ruído artificial nas imagens e avaliamos a capacidade de cada filtro em recuperar a qualidade original.

### Tipos de ruído:
- **Ruído Gaussiano:** Aditivo, com média 0 e desvio padrão 50. Simula distorções contínuas de sensor.
- **Ruído Sal-e-Pimenta:** Impulsivo, com probabilidade de 5%. Simula pixels corrompidos aleatoriamente.


In [ ]:
def adicionar_ruido_gaussiano(imagem, media=0, std=50):
    """Adiciona ruido gaussiano aditivo a imagem."""
    ruido = np.random.normal(media, std, imagem.shape).astype('float32')
    ruidosa = np.clip(imagem.astype('float32') + ruido, 0, 255).astype('uint8')
    return ruidosa

def adicionar_ruido_sal_pimenta(imagem, prob=0.05):
    """Adiciona ruido sal-e-pimenta a imagem."""
    ruidosa = np.copy(imagem)
    total_pixels = imagem.size
    # Sal (branco)
    n_sal = int(np.ceil(prob * total_pixels * 0.5))
    coords_sal = [np.random.randint(0, d - 1, n_sal) for d in imagem.shape[:2]]
    ruidosa[coords_sal[0], coords_sal[1]] = 255
    # Pimenta (preto)
    n_pimenta = int(np.ceil(prob * total_pixels * 0.5))
    coords_pim = [np.random.randint(0, d - 1, n_pimenta) for d in imagem.shape[:2]]
    ruidosa[coords_pim[0], coords_pim[1]] = 0
    return ruidosa

# Gerando imagens ruidosas
img_gauss_noise = adicionar_ruido_gaussiano(img_grupo)
img_sp_noise    = adicionar_ruido_sal_pimenta(img_grupo)

plot_images(
    [img_grupo_rgb,
     cv2.cvtColor(img_gauss_noise, cv2.COLOR_BGR2RGB),
     cv2.cvtColor(img_sp_noise,    cv2.COLOR_BGR2RGB)],
    ['Original', 'Ruido Gaussiano (std=50)', 'Ruido Sal-e-Pimenta (p=5%)'],
    figsize=(15, 5)
)


## 3. Parte A — Item (c): Análise Quantitativa com PSNR e SSIM


In [ ]:
def calcular_metricas(original, filtrada):
    """Calcula PSNR e SSIM entre imagem original e filtrada."""
    psnr_val = cv2.PSNR(original, filtrada)
    cinza_orig = cv2.cvtColor(original,  cv2.COLOR_BGR2GRAY)
    cinza_filt = cv2.cvtColor(filtrada,  cv2.COLOR_BGR2GRAY)
    ssim_val, _ = ssim(cinza_orig, cinza_filt, full=True)
    return psnr_val, ssim_val

k = 5  # kernel fixo para comparacao quantitativa
kp = k if k % 2 == 1 else k + 1

resultados = {}

for nome_ruido, img_ruidosa in [('Gaussiano', img_gauss_noise), ('Sal-e-Pimenta', img_sp_noise)]:
    resultados[nome_ruido] = {}
    for nome_filtro, img_filtrada in [
        ('Media',    cv2.blur(img_ruidosa, (k, k))),
        ('Gaussiano',cv2.GaussianBlur(img_ruidosa, (kp, kp), 0)),
        ('Mediana',  cv2.medianBlur(img_ruidosa, kp)),
        ('Bilateral',cv2.bilateralFilter(img_ruidosa, k, 75, 75)),
    ]:
        psnr_v, ssim_v = calcular_metricas(img_grupo, img_filtrada)
        resultados[nome_ruido][nome_filtro] = {'PSNR': psnr_v, 'SSIM': ssim_v}

# Exibindo tabela de resultados
print(f"{'Ruido':<15} {'Filtro':<12} {'PSNR (dB)':<12} {'SSIM':<8}")
print("-" * 50)
for nome_ruido, filtros in resultados.items():
    for nome_filtro, metricas in filtros.items():
        print(f"{nome_ruido:<15} {nome_filtro:<12} {metricas['PSNR']:<12.2f} {metricas['SSIM']:<8.4f}")
    print()


In [ ]:
# Visualizacao comparativa dos filtros no ruido Gaussiano (k=5)
k, kp = 5, 5
imgs_gauss = [
    cv2.cvtColor(img_gauss_noise, cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.blur(img_gauss_noise, (k, k)), cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.GaussianBlur(img_gauss_noise, (kp, kp), 0), cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.medianBlur(img_gauss_noise, kp), cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.bilateralFilter(img_gauss_noise, k, 75, 75), cv2.COLOR_BGR2RGB),
]
titles_g = ['Ruidosa (Gaussiano)', 'Media k=5', 'Gaussiano k=5', 'Mediana k=5', 'Bilateral k=5']
plot_images(imgs_gauss, titles_g, figsize=(18, 4))

# Visualizacao comparativa dos filtros no ruido Sal-e-Pimenta (k=5)
imgs_sp = [
    cv2.cvtColor(img_sp_noise, cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.blur(img_sp_noise, (k, k)), cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.GaussianBlur(img_sp_noise, (kp, kp), 0), cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.medianBlur(img_sp_noise, kp), cv2.COLOR_BGR2RGB),
    cv2.cvtColor(cv2.bilateralFilter(img_sp_noise, k, 75, 75), cv2.COLOR_BGR2RGB),
]
titles_sp = ['Ruidosa (S&P)', 'Media k=5', 'Gaussiano k=5', 'Mediana k=5', 'Bilateral k=5']
plot_images(imgs_sp, titles_sp, figsize=(18, 4))


**Análise Crítica e Discussão Quantitativa:**

Os resultados confirmam as expectativas teóricas:

1. **Ruído Gaussiano:** O filtro Gaussiano e o de Média apresentaram desempenhos similares em PSNR, pois ambos fazem médias ponderadas. O Gaussiano, porém, preserva melhor a aparência subjetiva pela suavização progressiva. O filtro Bilateral destacou-se no SSIM por tentar manter as bordas intactas durante a filtragem.

2. **Ruído Sal-e-Pimenta:** O filtro de **Mediana** foi amplamente superior. Por ser um operador de ordem, ele descarta os valores extremos (0 e 255 do ruído impulsivo) e seleciona o valor central da vizinhança, eliminando perfeitamente os pixels corrompidos sem introduzir borramento. Os filtros lineares (Média e Gaussiano) espalharam o ruído impulsivo pela imagem, degradando ainda mais o resultado.

---

## 4. Conclusões — Parte A

- Para ruído **Gaussiano**: filtros lineares (Média, Gaussiano) são adequados; o Bilateral é a melhor escolha quando a preservação de bordas é prioritária.
- Para ruído **Sal-e-Pimenta**: o filtro de **Mediana** é a escolha canônica e amplamente superior.
- Kernels maiores sempre aumentam o borramento, independentemente do filtro escolhido — há um trade-off entre supressão de ruído e preservação de detalhes.
